In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip -q install -U transformers datasets peft accelerate evaluate scikit-learn pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 11.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
tenso

In [1]:
%%writefile centralized_distilbert_lora_sst2.py
import os
import re
import json
import time
import random
import argparse

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    PeftModel,
    PeftConfig,
)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def save_json(obj, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def load_json(path: str, default=None):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return default


def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable_params, total_params


def get_latest_checkpoint(checkpoint_root: str):
    if not os.path.exists(checkpoint_root):
        return None

    candidates = []
    for name in os.listdir(checkpoint_root):
        full_path = os.path.join(checkpoint_root, name)
        if os.path.isdir(full_path):
            m = re.match(r"^checkpoint-epoch-(\d+)$", name)
            if m:
                candidates.append((int(m.group(1)), full_path))

    if not candidates:
        return None

    candidates.sort(key=lambda x: x[0])
    return candidates[-1][1]


def remove_dir(path: str):
    if not os.path.exists(path):
        return
    for root, dirs, files in os.walk(path, topdown=False):
        for file in files:
            os.remove(os.path.join(root, file))
        for d in dirs:
            os.rmdir(os.path.join(root, d))
    os.rmdir(path)


def cleanup_old_checkpoints(checkpoint_root: str, keep_last_n: int = 2):
    if not os.path.exists(checkpoint_root):
        return

    candidates = []
    for name in os.listdir(checkpoint_root):
        full_path = os.path.join(checkpoint_root, name)
        if os.path.isdir(full_path):
            m = re.match(r"^checkpoint-epoch-(\d+)$", name)
            if m:
                candidates.append((int(m.group(1)), full_path))

    candidates.sort(key=lambda x: x[0])
    to_delete = candidates[:-keep_last_n]

    for _, path in to_delete:
        remove_dir(path)


def save_lora_model_and_tokenizer(model, tokenizer, save_dir: str):
    ensure_dir(save_dir)
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)


def load_lora_model_for_resume(checkpoint_dir: str, device: torch.device):
    peft_cfg = PeftConfig.from_pretrained(checkpoint_dir)
    base_model = AutoModelForSequenceClassification.from_pretrained(
        peft_cfg.base_model_name_or_path,
        num_labels=2
    )
    model = PeftModel.from_pretrained(base_model, checkpoint_dir)
    model.to(device)
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir, use_fast=True)
    return model, tokenizer, peft_cfg.base_model_name_or_path


def create_new_lora_model(model_name: str, lora_r: int, lora_alpha: int, lora_dropout: float, device: torch.device):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        inference_mode=False,
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=["q_lin", "v_lin"],
        bias="none",
        modules_to_save=["classifier", "pre_classifier"],
    )

    model = get_peft_model(base_model, lora_config)
    model.to(device)
    return model, tokenizer


def evaluate_model(model, dataloader, device):
    model.eval()
    losses = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            logits = outputs.logits

            losses.append(loss.item())
            preds = torch.argmax(logits, dim=-1)

            all_preds.extend(preds.detach().cpu().numpy().tolist())
            all_labels.extend(batch["labels"].detach().cpu().numpy().tolist())

    eval_loss = float(np.mean(losses)) if losses else 0.0
    accuracy = float(accuracy_score(all_labels, all_preds))
    macro_f1 = float(f1_score(all_labels, all_preds, average="macro"))

    return {
        "eval_loss": eval_loss,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
    }


def train_one_epoch(model, dataloader, optimizer, scheduler, device, use_amp=False):
    model.train()
    losses = []

    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=use_amp):
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        losses.append(loss.item())

    train_loss = float(np.mean(losses)) if losses else 0.0
    return train_loss


def save_checkpoint(
    checkpoint_root,
    epoch,
    model,
    tokenizer,
    optimizer,
    scheduler,
    history,
    trainer_state,
):
    ckpt_dir = os.path.join(checkpoint_root, f"checkpoint-epoch-{epoch}")
    ensure_dir(ckpt_dir)

    save_lora_model_and_tokenizer(model, tokenizer, ckpt_dir)
    torch.save(optimizer.state_dict(), os.path.join(ckpt_dir, "optimizer.pt"))
    torch.save(scheduler.state_dict(), os.path.join(ckpt_dir, "scheduler.pt"))
    save_json(history, os.path.join(ckpt_dir, "history.json"))
    save_json(trainer_state, os.path.join(ckpt_dir, "trainer_state.json"))

    cleanup_old_checkpoints(checkpoint_root, keep_last_n=2)
    return ckpt_dir


def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--model_name", type=str, default="distilbert-base-uncased")
    parser.add_argument("--dataset_name", type=str, default="glue")
    parser.add_argument("--dataset_config", type=str, default="sst2")
    parser.add_argument("--output_dir", type=str, required=True)

    parser.add_argument("--max_length", type=int, default=256)
    parser.add_argument("--train_batch_size", type=int, default=16)
    parser.add_argument("--eval_batch_size", type=int, default=32)
    parser.add_argument("--learning_rate", type=float, default=2e-4)
    parser.add_argument("--weight_decay", type=float, default=0.01)
    parser.add_argument("--warmup_ratio", type=float, default=0.1)

    parser.add_argument("--lora_r", type=int, default=8)
    parser.add_argument("--lora_alpha", type=int, default=16)
    parser.add_argument("--lora_dropout", type=float, default=0.1)

    parser.add_argument("--patience", type=int, default=3)
    parser.add_argument("--metric_for_best_model", type=str, default="accuracy", choices=["accuracy", "macro_f1"])
    parser.add_argument("--seed", type=int, default=42)

    args = parser.parse_args()

    set_seed(args.seed)

    output_dir = args.output_dir
    checkpoint_root = os.path.join(output_dir, "checkpoints")
    best_model_dir = os.path.join(output_dir, "best_model")
    final_model_dir = os.path.join(output_dir, "final_model")
    logs_dir = os.path.join(output_dir, "logs")

    ensure_dir(output_dir)
    ensure_dir(checkpoint_root)
    ensure_dir(best_model_dir)
    ensure_dir(final_model_dir)
    ensure_dir(logs_dir)

    history_json_path = os.path.join(logs_dir, "history.json")
    history_csv_path = os.path.join(logs_dir, "history.csv")
    run_info_json_path = os.path.join(output_dir, "run_info.json")
    final_summary_json_path = os.path.join(output_dir, "final_summary.json")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = torch.cuda.is_available()

    print("=" * 100)
    print("Centralized DistilBERT + LoRA on SST-2")
    print(f"Device           : {device}")
    print(f"AMP enabled      : {use_amp}")
    print(f"Output dir       : {output_dir}")
    print(f"Checkpoints dir  : {checkpoint_root}")
    print(f"Best model dir   : {best_model_dir}")
    print(f"Final model dir  : {final_model_dir}")
    print(f"Logs dir         : {logs_dir}")
    print("=" * 100)

    latest_ckpt = get_latest_checkpoint(checkpoint_root)

    if latest_ckpt is not None:
        print(f"Resume from checkpoint: {latest_ckpt}")
        model, tokenizer, base_model_name = load_lora_model_for_resume(latest_ckpt, device)
        history = load_json(os.path.join(latest_ckpt, "history.json"), default=[])
        trainer_state = load_json(os.path.join(latest_ckpt, "trainer_state.json"), default={})
        start_epoch = int(trainer_state.get("epoch", 0)) + 1
        best_metric_so_far = float(trainer_state.get("best_metric_so_far", -1.0))
        patience_counter = int(trainer_state.get("patience_counter", 0))
    else:
        print("No checkpoint found. Start training from scratch.")
        model, tokenizer = create_new_lora_model(
            model_name=args.model_name,
            lora_r=args.lora_r,
            lora_alpha=args.lora_alpha,
            lora_dropout=args.lora_dropout,
            device=device,
        )
        base_model_name = args.model_name
        history = []
        start_epoch = 1
        best_metric_so_far = -1.0
        patience_counter = 0

    trainable_params, total_params = count_parameters(model)
    setting = "centralized_distilbert_lora"

    raw_dataset = load_dataset(args.dataset_name, args.dataset_config)
    train_ds = raw_dataset["train"]
    val_ds = raw_dataset["validation"]

    def tokenize_fn(examples):
        return tokenizer(
            examples["sentence"],
            truncation=True,
            padding=False,
            max_length=args.max_length,
        )

    train_ds = train_ds.map(tokenize_fn, batched=True, desc="Tokenizing train")
    val_ds = val_ds.map(tokenize_fn, batched=True, desc="Tokenizing validation")

    keep_cols = ["input_ids", "attention_mask", "label"]
    train_ds = train_ds.remove_columns(
        [c for c in train_ds.column_names if c not in keep_cols]
    ).rename_column("label", "labels")
    val_ds = val_ds.remove_columns(
        [c for c in val_ds.column_names if c not in keep_cols]
    ).rename_column("label", "labels")

    train_ds.set_format(type="torch")
    val_ds.set_format(type="torch")

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8 if torch.cuda.is_available() else None)

    train_loader = DataLoader(
        train_ds,
        batch_size=args.train_batch_size,
        shuffle=True,
        collate_fn=data_collator,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=args.eval_batch_size,
        shuffle=False,
        collate_fn=data_collator,
        pin_memory=torch.cuda.is_available(),
    )

    steps_per_epoch = len(train_loader)
    if steps_per_epoch == 0:
        raise ValueError("Train loader is empty.")

    virtual_max_epochs = 1000
    total_training_steps = steps_per_epoch * virtual_max_epochs
    warmup_steps = int(total_training_steps * args.warmup_ratio)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=args.learning_rate,
        weight_decay=args.weight_decay,
    )
    scheduler = get_linear_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_training_steps,
    )

    if latest_ckpt is not None:
        opt_path = os.path.join(latest_ckpt, "optimizer.pt")
        sch_path = os.path.join(latest_ckpt, "scheduler.pt")
        if os.path.exists(opt_path):
            optimizer.load_state_dict(torch.load(opt_path, map_location=device))
        if os.path.exists(sch_path):
            scheduler.load_state_dict(torch.load(sch_path, map_location=device))

    run_info = {
        "task_type": "text_classification",
        "learning_type": "centralized",
        "model_name": args.model_name,
        "base_model_name": base_model_name,
        "dataset_name": "glue",
        "dataset_config": "sst2",
        "setting": setting,
        "peft_method": "LoRA",
        "lora_r": args.lora_r,
        "lora_alpha": args.lora_alpha,
        "lora_dropout": args.lora_dropout,
        "output_dir": output_dir,
        "checkpoints_dir": checkpoint_root,
        "best_model_dir": best_model_dir,
        "final_model_dir": final_model_dir,
        "logs_dir": logs_dir,
        "history_json": history_json_path,
        "history_csv": history_csv_path,
        "client_history_json": None,
        "client_history_csv": None,
        "note": "Centralized training, so no client logs are generated.",
    }
    save_json(run_info, run_info_json_path)

    print(f"Train samples      : {len(train_ds)}")
    print(f"Validation samples : {len(val_ds)}")
    print(f"Trainable params   : {trainable_params:,}")
    print(f"Total params       : {total_params:,}")
    print(f"Metric for best    : {args.metric_for_best_model}")
    print(f"Patience           : {args.patience}")
    print(f"Start epoch        : {start_epoch}")

    stopped_early = False
    last_completed_epoch = start_epoch - 1
    epoch = start_epoch

    while True:
        print("\n" + "=" * 100)
        print(f"Epoch {epoch}")
        print("=" * 100)

        epoch_start = time.time()

        train_loss = train_one_epoch(
            model=model,
            dataloader=train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            use_amp=use_amp,
        )

        eval_metrics = evaluate_model(model, val_loader, device=device)
        epoch_time = time.time() - epoch_start

        current_metric = float(eval_metrics[args.metric_for_best_model])
        is_new_best = current_metric > best_metric_so_far

        if is_new_best:
            best_metric_so_far = current_metric
            patience_counter = 0
            save_lora_model_and_tokenizer(model, tokenizer, best_model_dir)
        else:
            patience_counter += 1

        row = {
            "epoch": int(epoch),
            "train_loss": float(train_loss),
            "eval_loss": float(eval_metrics["eval_loss"]),
            "accuracy": float(eval_metrics["accuracy"]),
            "macro_f1": float(eval_metrics["macro_f1"]),
            "time_per_epoch": float(epoch_time),
            "trainable_params": int(trainable_params),
            "total_params": int(total_params),
            "model_name": str(args.model_name),
            "dataset_name": "sst2",
            "setting": str(setting),
            "best_metric_so_far": float(best_metric_so_far),
            "patience_counter": int(patience_counter),
            "is_new_best": bool(is_new_best),
        }
        history.append(row)

        pd.DataFrame(history).to_csv(history_csv_path, index=False)
        save_json(history, history_json_path)

        trainer_state = {
            "epoch": int(epoch),
            "best_metric_so_far": float(best_metric_so_far),
            "patience_counter": int(patience_counter),
            "metric_for_best_model": args.metric_for_best_model,
            "stopped_early": False,
            "model_name": args.model_name,
            "dataset_name": "sst2",
            "setting": setting,
            "lora_r": args.lora_r,
            "lora_alpha": args.lora_alpha,
            "lora_dropout": args.lora_dropout,
        }

        ckpt_dir = save_checkpoint(
            checkpoint_root=checkpoint_root,
            epoch=epoch,
            model=model,
            tokenizer=tokenizer,
            optimizer=optimizer,
            scheduler=scheduler,
            history=history,
            trainer_state=trainer_state,
        )

        print(f"train_loss         = {row['train_loss']:.6f}")
        print(f"eval_loss          = {row['eval_loss']:.6f}")
        print(f"accuracy           = {row['accuracy']:.6f}")
        print(f"macro_f1           = {row['macro_f1']:.6f}")
        print(f"time_per_epoch     = {row['time_per_epoch']:.2f} sec")
        print(f"best_metric_so_far = {row['best_metric_so_far']:.6f}")
        print(f"patience_counter   = {row['patience_counter']}")
        print(f"is_new_best        = {row['is_new_best']}")
        print(f"saved_checkpoint   = {ckpt_dir}")

        last_completed_epoch = epoch

        if patience_counter >= args.patience:
            print(f"Early stopping triggered after {args.patience} consecutive non-improving epoch(s).")
            stopped_early = True
            break

        epoch += 1

    save_lora_model_and_tokenizer(model, tokenizer, final_model_dir)

    best_model, _, _ = load_lora_model_for_resume(best_model_dir, device)
    best_val_metrics = evaluate_model(best_model, val_loader, device=device)

    final_summary = {
        "model_name": args.model_name,
        "dataset_name": "sst2",
        "setting": setting,
        "learning_type": "centralized",
        "peft_method": "LoRA",
        "lora_r": args.lora_r,
        "lora_alpha": args.lora_alpha,
        "lora_dropout": args.lora_dropout,
        "output_dir": output_dir,
        "checkpoints_dir": checkpoint_root,
        "best_model_dir": best_model_dir,
        "final_model_dir": final_model_dir,
        "logs_dir": logs_dir,
        "history_json": history_json_path,
        "history_csv": history_csv_path,
        "client_history_json": None,
        "client_history_csv": None,
        "total_epochs_completed": int(last_completed_epoch),
        "early_stopped": bool(stopped_early),
        "metric_for_best_model": args.metric_for_best_model,
        "best_metric_so_far": float(best_metric_so_far),
        "best_validation_metrics": {
            "eval_loss": float(best_val_metrics["eval_loss"]),
            "accuracy": float(best_val_metrics["accuracy"]),
            "macro_f1": float(best_val_metrics["macro_f1"]),
        },
        "history_columns": [
            "epoch",
            "train_loss",
            "eval_loss",
            "accuracy",
            "macro_f1",
            "time_per_epoch",
            "trainable_params",
            "total_params",
            "model_name",
            "dataset_name",
            "setting",
            "best_metric_so_far",
            "patience_counter",
            "is_new_best",
        ],
        "generated_files_and_dirs": [
            "checkpoints/checkpoint-epoch-*",
            "best_model/",
            "final_model/",
            "logs/history.json",
            "logs/history.csv",
            "run_info.json",
            "final_summary.json",
        ],
        "note": "This is centralized training. Therefore no client_history.json/csv is generated."
    }
    save_json(final_summary, final_summary_json_path)

    print("\n" + "=" * 100)
    print("TRAINING FINISHED")
    print("=" * 100)
    print(f"Output dir         : {output_dir}")
    print(f"Checkpoints dir    : {checkpoint_root}")
    print(f"Best model dir     : {best_model_dir}")
    print(f"Final model dir    : {final_model_dir}")
    print(f"Logs dir           : {logs_dir}")
    print(f"history.json       : {history_json_path}")
    print(f"history.csv        : {history_csv_path}")
    print(f"run_info.json      : {run_info_json_path}")
    print(f"final_summary.json : {final_summary_json_path}")
    print("Generated:")
    print("- checkpoints/checkpoint-epoch-*   (only 2 latest checkpoints kept)")
    print("- best_model/")
    print("- final_model/")
    print("- logs/history.json")
    print("- logs/history.csv")
    print("- run_info.json")
    print("- final_summary.json")
    print("\nSelf-check:")
    print("✓ Full runnable Colab Python script")
    print("✓ Save all outputs to Google Drive/MyDrive")
    print("✓ Automatic resume from latest checkpoint")
    print("✓ Keep only 2 latest checkpoints")
    print("✓ Early stopping with patience=3")
    print("✓ Save best_model and final_model separately")
    print("✓ Export history.json and history.csv")
    print("✓ History columns complete")
    print("✓ No client logs because this is centralized, not federated")


if __name__ == "__main__":
    main()

Writing centralized_distilbert_lora_sst2.py


In [2]:
!python centralized_distilbert_lora_sst2.py \
  --model_name distilbert-base-uncased \
  --dataset_name glue \
  --dataset_config sst2 \
  --output_dir "/content/drive/MyDrive/centralized_distilbert_lora_sst2"

Centralized DistilBERT + LoRA on SST-2
Device           : cuda
AMP enabled      : True
Output dir       : /content/drive/MyDrive/centralized_distilbert_lora_sst2
Checkpoints dir  : /content/drive/MyDrive/centralized_distilbert_lora_sst2/checkpoints
Best model dir   : /content/drive/MyDrive/centralized_distilbert_lora_sst2/best_model
Final model dir  : /content/drive/MyDrive/centralized_distilbert_lora_sst2/final_model
Logs dir         : /content/drive/MyDrive/centralized_distilbert_lora_sst2/logs
No checkpoint found. Start training from scratch.
config.json: 100% 483/483 [00:00<00:00, 1.41MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 74.3kB/s]
vocab.txt: 232kB [00:00, 4.99MB/s]
tokenizer.json: 466kB [00:00, 8.85MB/s]
model.safetensors: 100% 268M/268M [00:02<00:00, 96.8MB/s]
Loading weights: 100% 100/100 [00:00<00:00, 945.37it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-unca